In [ ]:
import random
random.seed(42)

import torch

import numpy as np
import pyomo.environ as pyo
import pandas as pd

from pyomo.environ import *
from utils import generate_simple_synthetic_data, const_embed
from trainer import Trainer

np.random.seed(43)

In [ ]:

X, y, true_mean, true_std = generate_simple_synthetic_data(n_samples=10)

print(X.head())
print(y.head())


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model_config = {
    'hidden_layers': 1,
    'hidden_size': 5,
    'drop': 0.2
    }

training_config = {
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 500,
    'patience': 20
    }

trainer = Trainer(X, y, model_config, training_config, device=device)
trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

trainer.train()

trainer.test()

In [ ]:
trainer.model

In [ ]:
cons = trainer.constraint_build()
cons.to_csv('constraints.csv')

In [ ]:
cons = pd.read_csv('constraints.csv', index_col=0)
cons

In [ ]:
n_sc = 3
contextual_var = trainer.X_test

In [ ]:
model = pyo.ConcreteModel()
    
model.N = pyo.RangeSet(0,0)
model.W = pyo.RangeSet(1,n_sc) # scenarios

pi = [1/n_sc for i in range(n_sc)]

model.x = pyo.Var(contextual_var.columns.sort_values(), model.N)

model.y = pyo.Var(pyo.Any, dense=False, domain=pyo.Reals)

model.rent = pyo.Var(model.W, within=pyo.Reals) # rentabilidad per scenario

#definition of the objective function
def obj_expression(model):
    return sum(pi[w-1] * model.rent[w] for w in model.W)
model.OBJ = pyo.Objective(rule=obj_expression, sense=pyo.maximize)

#definition of the constraints

def rent_expression(model, w):
    return model.rent[w] == sum(model.y['demand', n, w]+model.x['price',n] for n in model.N)
model.rent_expression = pyo.Constraint(model.W, rule=rent_expression)

In [ ]:
trainer.y_mean

In [ ]:
const_embed(model=trainer, opt_model=model, constaints=cons, outcome='demand', n_scenarios=n_sc)

In [ ]:
opt = pyo.SolverFactory('glpk')
results = opt.solve(model, tee=True)

In [ ]:
for c in model.component_objects(Constraint, active=True):
    print(f"Constraint component: {c.name}")
    for index in c:
        print(f"  {index}: {c[index].expr}")
